# 03 — Birleştirme ve sızıntısız split (Faz 3)

**Çalışma zamanı: CPU.** GPU gerekmez. Toplam süre ~1–1,5 saat; en uzun adımlar veriyi Drive'dan açmak (~15 dk), görüntüleri yeniden boyutlandırmak (~40 dk) ve zip'lemek (~5 dk).

Bu notebook kendi başına çalışır: veriyi Drive'daki yedeklerden kendisi geri yükler, 01/02'yi tekrar çalıştırmana gerek yok.

**Audit'ten çıkan kararlar**
- **FASDD resmi split'i kullanılmıyor.** Test görüntülerinin %93'ünün train/val'de aynı videodan bir karesi vardı. Görüntüler yakın-kopya gruplarına ayrılıyor (tüm veride phash ≤ 4 bit, FASDD içinde ≤ 8 bit); her grup bütün olarak tek bir split'e gidiyor. 4 bit tek başına yetmedi: 6–8 bit farklı çiftlerin aynı videonun kareleri olduğu gözle doğrulandı.
- **Boreal:** Heinola olayı tamamen test. Evo, Karkkila, Ruokolahti video bazında train/val. Dumanlı klasörde olup etiketi boş 5 görüntü çıkarılıyor (atlanmış etiket olabilir).
- **Bilinen sınır:** phash ~8 bitten sonra farklı sahneleri de benzer sayıyor (Boreal'deki farklı yangınlarla doğrulandı). 10–12 bitte kalan olası sızıntı 04'te DINOv2 benzerliğiyle temizlenecek.
- **D-Fire kullanılmıyor:** FASDD ile aynı görüntüleri içeriyor, FASDD'nin %48'i zaten negatif.
- **Görüntüler uzun kenarı 1280 px olacak şekilde küçültülüyor.** 1280'e kadarki eğitimlerde bilgi kaybı yok, eğitim çok hızlanır. Orijinaller Drive'da kalır.

**Çıktı:** `/content/firesmoke_v1` (eğitim buradan okur) ve Drive'da `firesmoke/datasets/firesmoke_v1.zip` (kalıcı kopya).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install imagehash tabulate

In [ ]:
import json, random, re, shutil, subprocess, time
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
import imagehash

ROOT = Path('/content/drive/MyDrive/firesmoke')   # Drive (kalıcı)
RAW  = Path('/content/raw')                       # ham veri (geçici)
OUT  = Path('/content/firesmoke_v1')              # yeni dataset (geçici)
REP  = ROOT / 'reports'

# --- Ayarlar -------------------------------------------------------------
SEED       = 0
RATIOS     = {'train': 0.70, 'val': 0.15, 'test': 0.15}   # FASDD için
BOREAL_VAL = 0.15          # Boreal train olaylarından val'e ayrılacak video oranı
TEST_EVENT = 'heinola'     # Boreal'de tamamen teste giden olay
MAX_SIDE   = 1280          # uzun kenar bundan büyükse küçültülür
THR_GROUP  = 4             # tüm görüntüler: phash farkı <= bu ise aynı grup
THR_FASDD  = 8             # FASDD içi: aynı videonun daha uzak kareleri (gözle doğrulandı)
THR_CHECK  = 12            # split sonrası gevşek kontrol; phash'in güvenilir sınırı ~8 bit
BIG_GROUP  = 300           # bundan büyük FASDD grupları train'e (test tek sahneye boğulmasın)
NAMES      = {0: 'fire', 1: 'smoke'}
IMG_EXT    = {'.jpg', '.jpeg', '.png', '.bmp'}
# -------------------------------------------------------------------------

RAW.mkdir(parents=True, exist_ok=True)
REP.mkdir(parents=True, exist_ok=True)
random.seed(SEED)

## 1. Ham veriyi geri yükle

Zaten açıksa (aynı oturumda 01'i çalıştırdıysan) atlanır. Değilse FASDD tar'ları ve Boreal zip'leri doğrudan Drive'dan `/content/raw`'a açılır. macOS hayalet dosyaları (`._*`, `__MACOSX`) silinir.

In [ ]:
def remove_mac_junk(base):
    junk = [p for p in base.rglob('._*') if p.is_file()]
    for p in junk:
        p.unlink()
    macosx = [p for p in base.rglob('__MACOSX') if p.is_dir()]
    for p in macosx:
        shutil.rmtree(p, ignore_errors=True)
    print(f'  {len(junk)} adet ._ dosyası, {len(macosx)} adet __MACOSX silindi')

def find_dir(base, name):
    hits = [p for p in base.rglob(name) if p.is_dir()]
    return min(hits, key=lambda p: len(p.parts)) if hits else None

F = RAW / 'fasdd_uav'
B = RAW / 'boreal'

t0 = time.time()
if find_dir(F, 'YOLO_UAV') is None:
    backup = ROOT / 'raw_zips' / 'fasdd_uav'
    F.mkdir(parents=True, exist_ok=True)
    for t in ['annotations.tar', 'images.tar']:
        assert (backup / t).exists(), f'{backup / t} yok — önce 01_download çalıştırılmalı'
        print('açılıyor:', t)
        subprocess.run(['tar', '-xf', str(backup / t), '-C', str(F)], check=True)
else:
    print('FASDD zaten açık')
remove_mac_junk(F)

if find_dir(B, 'Boreal-Forest-Fire-Subset-A') is None:
    zips = sorted((ROOT / 'raw_zips').glob('*.zip'))
    assert zips, 'raw_zips/ içinde Boreal zip bulunamadı'
    B.mkdir(parents=True, exist_ok=True)
    for z in zips:
        print('açılıyor:', z.name)
        subprocess.run(['unzip', '-q', '-o', str(z), '-d', str(B)], check=True)
else:
    print('Boreal zaten açık')
remove_mac_junk(B)
print(f'süre: {time.time() - t0:.0f} sn')

## 2. Envanter

Her görüntü bir satır: kaynak, olay/video, orijinal split, etiketler. Etiketler okunurken temizlenir:
- sıfır alanlı kutular atılır (audit'te 1 tane vardı),
- koordinatlar 0–1 aralığına kırpılır,
- Boreal'de duman `0` → bizim şemamızda `1`.

In [ ]:
def read_yolo(path, remap=None):
    """YOLO etiketini oku; (boxes, dropped) döndür."""
    boxes, dropped = [], 0
    if path is None or not path.exists():
        return boxes, dropped
    for line in path.read_text().splitlines():
        parts = line.split()
        if len(parts) != 5:
            continue
        c = int(float(parts[0]))
        if remap is not None:
            c = remap[c]
        x, y, w, h = map(float, parts[1:])
        x0, y0 = max(0.0, x - w / 2), max(0.0, y - h / 2)
        x1, y1 = min(1.0, x + w / 2), min(1.0, y + h / 2)
        if x1 - x0 <= 0 or y1 - y0 <= 0:
            dropped += 1
            continue
        boxes.append((c, (x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0))
    return boxes, dropped

rows, dropped_total = [], 0

# --- FASDD_UAV
IMG_DIR  = find_dir(F, 'images')
YOLO_DIR = find_dir(F, 'YOLO_UAV')
orig_split = {}
for s in ['train', 'val', 'test']:
    for line in (YOLO_DIR / f'{s}.txt').read_text().split():
        orig_split[Path(line).stem] = s
for p in sorted(IMG_DIR.rglob('*')):
    if p.suffix.lower() not in IMG_EXT:
        continue
    bx, d = read_yolo(YOLO_DIR / 'labels' / f'{p.stem}.txt')
    dropped_total += d
    rows.append({'uid': f'fasdd_{p.stem}', 'source': 'fasdd', 'path': p, 'boxes': bx,
                 'event': '', 'video': '', 'orig_split': orig_split.get(p.stem, '?')})

# --- Boreal Subset A
SA = find_dir(B, 'Boreal-Forest-Fire-Subset-A')
lbl_index = {p.stem: p for d in SA.glob('*-Labels') for p in d.rglob('*.txt')}
pat = re.compile(r'^(?P<event>[A-Za-z]+)_(?P<video>.+?)_frame\d+$')
unparsed = []
for d in sorted(SA.glob('*-Images')):
    for p in sorted(d.rglob('*')):
        if p.suffix.lower() not in IMG_EXT:
            continue
        m = pat.match(p.stem)
        if m:
            event, video = m['event'].lower(), f"{m['event'].lower()}_{m['video']}"
        else:
            event = p.stem.split('_')[0].lower()
            video = p.stem            # video bulunamazsa görüntü kendi grubunu oluşturur
            unparsed.append(p.name)
        bx, dd = read_yolo(lbl_index.get(p.stem), remap={0: 1})
        dropped_total += dd
        rows.append({'uid': f'boreal_{p.stem}', 'source': 'boreal', 'path': p, 'boxes': bx,
                     'event': event, 'video': video, 'orig_split': ''})

df = pd.DataFrame(rows)
df['n_fire']  = df.boxes.map(lambda b: sum(c == 0 for c, *_ in b))
df['n_smoke'] = df.boxes.map(lambda b: sum(c == 1 for c, *_ in b))
df['cat'] = np.select([(df.n_fire > 0) & (df.n_smoke > 0), df.n_fire > 0, df.n_smoke > 0],
                      ['fire+smoke', 'sadece fire', 'sadece smoke'], 'negatif')
assert df.uid.is_unique, 'aynı isimli iki görüntü var'

# Dumanlı klasörde (Empty-* dışında) olup etiketi boş Boreal görüntüleri: atlanmış etiket olabilir
in_empty = df.path.map(lambda p: any(part.startswith('Empty') for part in p.parts))
sus = df[(df.source == 'boreal') & (df.cat == 'negatif') & ~in_empty]
df = df.drop(sus.index).reset_index(drop=True)
print('etiketi boş olduğu için çıkarılan Boreal görüntüsü:', len(sus))

print('atılan sıfır alanlı kutu:', dropped_total)
print('olay/video okunamayan Boreal dosyası:', len(unparsed), unparsed[:3])
print('Boreal\'de duman dışı sınıf:', int((df[df.source == 'boreal'].n_fire > 0).sum()))
display(df.groupby('source').cat.value_counts().unstack(fill_value=0))
b = df[df.source == 'boreal']
display(pd.DataFrame({'görüntü': b.groupby('event').size(),
                      'negatif': b[b.cat == 'negatif'].groupby('event').size(),
                      'video': b.groupby('event').video.nunique()}).fillna(0).astype(int))

Kontrol: FASDD 25.097 görüntü olmalı. Boreal dosya adlarından sayılır; v1'de Evo 931, Heinola 1.123 (217 boş kare), Karkkila 1.133, Ruokolahti 1.767 çıktı. `image_counts.txt`'teki "Total counts" bölümünde Heinola ile Karkkila'nın toplamları karışık; dosya adları esas alınır. Evo'nun `evodji` diye görünmesi dosya adlandırmasından gelir, zararsızdır.

## 3. Yakın-kopya grupları

phash, audit'teki hızlı yöntemle hesaplanır (~5–10 dk). Üç tür bağlantı aynı gruba sokar:
- tüm veride phash farkı ≤ `THR_GROUP` (4 bit),
- FASDD içinde phash farkı ≤ `THR_FASDD` (8 bit): aynı videonun daha uzak kareleri. Boreal'e uygulanmıyor; farklı yangınlar 8 bitte birbirine bağlanabiliyor ve Heinola testine başka olay sızardı.
- Boreal'de aynı video.

Birbirine zincirle bağlı her şey tek grup olur; grup hiçbir zaman iki split'e bölünmez.

In [ ]:
def ph(p):
    try:
        with Image.open(p) as im:
            im.draft('L', (256, 256))
            return int(str(imagehash.phash(im)), 16)
    except Exception:
        return None

t0 = time.time()
with ThreadPoolExecutor(8) as ex:
    df['phash'] = list(ex.map(ph, df.path))
print(f'phash: {time.time() - t0:.0f} sn, hesaplanamayan: {int(df.phash.isna().sum())}')
df = df[df.phash.notna()].reset_index(drop=True)
H = df.phash.astype('uint64').to_numpy()

_POP = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)
def popcount64(x):
    if hasattr(np, 'bitwise_count'):
        return np.bitwise_count(x)
    x = np.ascontiguousarray(x)
    return _POP[x.view(np.uint8).reshape(*x.shape, 8)].sum(-1)

def near_pairs(idx_a, idx_b, thr, chunk=512):
    """idx_a x idx_b arasında hamming <= thr olan (a, b, dist) çiftleri."""
    out, Hb = [], H[idx_b]
    for k in range(0, len(idx_a), chunk):
        ia = idx_a[k:k + chunk]
        d = popcount64(H[ia][:, None] ^ Hb[None, :])
        r, c = np.nonzero(d <= thr)
        out.append(np.stack([ia[r], idx_b[c], d[r, c]], axis=1))
    return np.concatenate(out) if out else np.zeros((0, 3), dtype=np.int64)

In [ ]:
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

all_idx = np.arange(len(df))
pairs = near_pairs(all_idx, all_idx, THR_GROUP)
pairs = pairs[pairs[:, 0] < pairs[:, 1]]

# FASDD içi 8 bit komşuluk
f_idx = df.index[df.source == 'fasdd'].to_numpy()
p8 = near_pairs(f_idx, f_idx, THR_FASDD)
p8 = p8[p8[:, 0] < p8[:, 1]]

# Boreal: aynı videonun karelerini zincirle
vid_edges = []
for _, g in df[df.source == 'boreal'].groupby('video'):
    ix = g.index.to_numpy()
    vid_edges += list(zip(ix[:-1], ix[1:]))
vid_edges = np.array(vid_edges, dtype=np.int64).reshape(-1, 2)

e = np.concatenate([pairs[:, :2], p8[:, :2], vid_edges])
graph = coo_matrix((np.ones(len(e)), (e[:, 0], e[:, 1])), shape=(len(df), len(df)))
n_groups, df['group'] = connected_components(graph, directed=False)

gsize = df.group.value_counts()
mixed_src = df.groupby('group').source.nunique()
print('phash çifti (4 bit):', len(pairs), '  FASDD 8 bit çifti:', len(p8))
print('grup:', n_groups, '  tek başına:', int((gsize == 1).sum()))
print('en büyük 10 grup:', gsize.head(10).tolist())
print('hem FASDD hem Boreal içeren grup:', int((mixed_src > 1).sum()))

## 4. Grupları split'lere dağıt

- **Heinola** içeren her grup → test.
- **Diğer Boreal grupları** → train / val (videoların ~%15'i val).
- **FASDD grupları** → %70 / %15 / %15. `BIG_GROUP`'tan büyük gruplar train'e sabitlenir; testin tek bir sahneye boğulmasını önler. Kalanlar, her split'te dört kategorinin (fire+smoke, sadece fire, sadece smoke, negatif) oranı korunacak şekilde dağıtılır. Büyükten küçüğe giden açgözlü bir dengeleme kullanılıyor.

In [ ]:
CATS = ['fire+smoke', 'sadece fire', 'sadece smoke', 'negatif']

def assign(groups, ratios, forced=None, seed=SEED):
    """groups: DataFrame (index=group, columns=CATS). Her gruba bir split döndür."""
    splits = list(ratios)
    tot = groups[CATS].sum().to_numpy(float) + 1e-9
    target = np.array([[ratios[s] * t for t in tot] for s in splits])
    cur = np.zeros_like(target)
    out = {}
    forced = forced or {}
    for gid, s in forced.items():
        cur[splits.index(s)] += groups.loc[gid, CATS].to_numpy(float)
        out[gid] = s
    rest = groups.drop(index=list(forced))
    order = rest.sample(frac=1, random_state=seed)
    order = order.loc[order[CATS].sum(axis=1).sort_values(ascending=False, kind='stable').index]
    for gid, row in order.iterrows():
        v = row[CATS].to_numpy(float)
        cost = [(((cur[k] + v - target[k]) / tot) ** 2).sum() - (((cur[k] - target[k]) / tot) ** 2).sum()
                for k in range(len(splits))]
        k = int(np.argmin(cost))
        cur[k] += v
        out[gid] = splits[k]
    return out

gcat = df.pivot_table(index='group', columns='cat', values='uid', aggfunc='count', fill_value=0)
gcat = gcat.reindex(columns=CATS, fill_value=0)
g_has_boreal = df.groupby('group').source.agg(lambda s: (s == 'boreal').any())
g_has_test   = df.groupby('group').event.agg(lambda s: (s == TEST_EVENT).any())

split_of_group = {}
for gid in g_has_test[g_has_test].index:
    split_of_group[gid] = 'test'

b_groups = gcat.loc[g_has_boreal & ~g_has_test]
split_of_group.update(assign(b_groups, {'train': 1 - BOREAL_VAL, 'val': BOREAL_VAL}))

f_groups = gcat.loc[~g_has_boreal & ~g_has_test]
big = f_groups[f_groups.sum(axis=1) > BIG_GROUP].index
split_of_group.update(assign(f_groups, RATIOS, forced={g: 'train' for g in big}))

df['split'] = df.group.map(split_of_group)
assert df.split.notna().all()
assert df.groupby('group').split.nunique().max() == 1, 'bir grup iki split\'e dağılmış'
print('train\'e sabitlenen büyük FASDD grubu:', len(big), '  görüntü:', int(f_groups.loc[big].sum().sum()))

t = df.pivot_table(index=['source', 'split'], columns='cat', values='uid', aggfunc='count', fill_value=0)
t = t.reindex(columns=CATS, fill_value=0)
t['toplam'] = t.sum(axis=1)
t['negatif %'] = (100 * t['negatif'] / t['toplam']).round(1)
display(t)
display(df.groupby('split').size().to_frame('görüntü').assign(oran=lambda x: (x.görüntü / x.görüntü.sum()).round(3)))

Kontrol: FASDD'de train/val/test oranları yaklaşık 0.70/0.15/0.15 olmalı, negatif oranı üç split'te birbirine yakın olmalı. Boreal'de test sadece Heinola içermeli.

## 5. Sızıntı doğrulaması

Üç eşikte, kaynağa göre ölçülür:
- **FASDD, ≤ 8 bit: 0 olmalı** (yapı gereği).
- **Boreal, ≤ 8 bit:** Heinola ile diğer olaylar arasında birkaç çift çıkabilir (v1'de 8). Farklı yangınlar oldukları için sızıntı değil.
- **≤ 12 bit:** phash burada farklı sahneleri de benzer sayıyor. v1'de FASDD testinin ~%35'inin 10–12 bitte komşusu vardı; 8 örnekten 4'ü gerçekten aynı sahneydi. Bu kalan kısım 04'te DINOv2 ile temizlenecek.

İki grid çizilir: FASDD'nin 8–12 bit çiftleri ve **referans** olarak Boreal'in farklı yangınlardan çiftleri (kesin sızıntı değil). İkisini karşılaştır.

In [ ]:
src = df.source.to_numpy()
def idx(s):
    return df.index[df.split == s].to_numpy()

def leak_pairs(a, b):
    p = pd.DataFrame(near_pairs(a, b, THR_CHECK), columns=['i', 'j', 'dist'])
    p['src_i'], p['src_j'] = src[p.i], src[p.j]
    return p

leak     = leak_pairs(idx('test'), np.concatenate([idx('train'), idx('val')]))
leak_val = leak_pairs(idx('val'), idx('train'))
for name, p in [('test ↔ train/val', leak), ('val  ↔ train    ', leak_val)]:
    for thr in [THR_GROUP, THR_FASDD, THR_CHECK]:
        q = p[p.dist <= thr]
        f = q[q.src_i == 'fasdd']
        print(f'{name} fark <= {thr:>2}: FASDD {len(f):>6} çift ({f.i.nunique()} görüntü) · '
              f'Boreal {int((q.src_i == "boreal").sum())} çift')

In [ ]:
def show_pairs(pr, title, n=8):
    if len(pr) == 0:
        print('çift yok:', title); return
    pr = pr.sort_values('dist').head(200).sample(min(n, len(pr)), random_state=SEED)
    fig, axes = plt.subplots(len(pr), 2, figsize=(9, 2.6 * len(pr)), squeeze=False)
    for row, r in zip(axes, pr.itertuples()):
        for ax, k in [(row[0], r.i), (row[1], r.j)]:
            with Image.open(df.path[k]) as im:
                im.draft('RGB', (512, 512))
                ax.imshow(im.convert('RGB'))
            ax.set_title(f'{df.split[k]} · {df.uid[k][:30]} · fark={r.dist}', fontsize=7)
            ax.axis('off')
    fig.suptitle(title)
    plt.tight_layout(); plt.show()

show_pairs(leak[(leak.src_i == 'fasdd') & (leak.dist > THR_FASDD)],
           f'FASDD test ↔ train/val, {THR_FASDD} < fark ≤ {THR_CHECK}')
show_pairs(leak[(leak.src_i == 'boreal') & (leak.dist > THR_FASDD)],
           f'REFERANS — Boreal, farklı yangınlar (kesin sızıntı değil), {THR_FASDD} < fark ≤ {THR_CHECK}')

Her satır bir çift: solda test görüntüsü, sağda train/val'deki en benzer komşusu. FASDD satırları referanstaki gibi "benzer ama farklı yer" görünüyorsa split temiz. Belirgin şekilde aynı sahneyse, bu notebook'un yapabileceğinin sınırı bu; kalan kısım 04'te temizlenir.

## 6. Dataset'i yaz

Her görüntü `/content/firesmoke_v1/images/<split>/` altına, kaynak önekiyle (`fasdd_`, `boreal_`) yazılır. Uzun kenarı `MAX_SIDE`'tan büyükse EXIF yönü düzeltilip küçültülür; YOLO koordinatları normalize olduğu için etiketler değişmez. Süre: ücretsiz Colab CPU'sunda ~40 dk; her 3.000 görüntüde ilerleme yazar. **Yarıda kesme:** tekrar çalıştırınca klasörü silip baştan başlar.

In [ ]:
if OUT.exists():
    shutil.rmtree(OUT)
for s in ['train', 'val', 'test']:
    (OUT / 'images' / s).mkdir(parents=True)
    (OUT / 'labels' / s).mkdir(parents=True)

def write_one(k):
    r = df.loc[k]
    dst = OUT / 'images' / r.split / f'{r.uid}.jpg'
    with Image.open(r.path) as im:
        W, H = im.size
        if max(W, H) > MAX_SIDE or r.path.suffix.lower() not in {'.jpg', '.jpeg'}:
            im.draft('RGB', (MAX_SIDE, MAX_SIDE))
            im = ImageOps.exif_transpose(im).convert('RGB')
            im.thumbnail((MAX_SIDE, MAX_SIDE), Image.LANCZOS)
            im.save(dst, quality=95)
            W, H = im.size
        else:
            shutil.copy2(r.path, dst)
    lines = [f'{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}' for c, x, y, w, h in r.boxes]
    (OUT / 'labels' / r.split / f'{r.uid}.txt').write_text('\n'.join(lines))
    return W, H

t0, wh = time.time(), []
with ThreadPoolExecutor(8) as ex:
    for n, r in enumerate(ex.map(write_one, df.index), 1):
        wh.append(r)
        if n % 3000 == 0:
            print(f'{n}/{len(df)}  {time.time() - t0:.0f} sn')
df['W'], df['H'] = zip(*wh)
print(f'yazıldı: {len(df)} görüntü, {time.time() - t0:.0f} sn')
print('uzun kenarı MAX_SIDE\'tan büyük kalan:', int((df[['W', 'H']].max(axis=1) > MAX_SIDE).sum()))

In [ ]:
df['file'] = 'images/' + df.split + '/' + df.uid + '.jpg'
manifest = df[['file', 'uid', 'source', 'split', 'group', 'event', 'video', 'orig_split',
               'cat', 'n_fire', 'n_smoke', 'W', 'H']].copy()
manifest['orig_path'] = df.path.map(lambda p: str(p.relative_to(RAW)))
manifest['phash'] = df.phash.map(lambda h: f'{int(h):016x}')
manifest.to_csv(OUT / 'manifest.csv', index=False)

def yaml_text(test='images/test'):
    return (f'path: {OUT}\ntrain: images/train\nval: images/val\ntest: {test}\n'
            'names:\n  0: fire\n  1: smoke\n')

(OUT / 'data.yaml').write_text(yaml_text())
for src in ['fasdd', 'boreal']:
    files = df[(df.split == 'test') & (df.source == src)].file
    (OUT / f'test_{src}.txt').write_text('\n'.join(f'./{f}' for f in files) + '\n')
    (OUT / f'data_test_{src}.yaml').write_text(yaml_text(f'test_{src}.txt'))
print((OUT / 'data.yaml').read_text())
print(sorted(p.name for p in OUT.iterdir()))

`data_test_fasdd.yaml` ve `data_test_boreal.yaml` Faz 6 için: test setini iki kaynakta ayrı ölçmeye yarar. Boreal testi (Heinola) modelin hiç görmediği bir yangın olayıdır; genelleme ölçümü odur.

## 7. Son veri üzerinde istatistik

Audit'in eksik kalan kısmı burada, kesinleşmiş dataset üzerinde: split × kaynak tablosu, kutu boyutları, örnek grid'ler.

In [ ]:
bx = pd.DataFrame([(k, c, w, h) for k, b in zip(df.index, df.boxes) for c, x, y, w, h in b],
                  columns=['k', 'cls', 'w', 'h'])
bx = bx.join(df[['split', 'source', 'W', 'H']], on='k')
bx['name'] = bx.cls.map(NAMES)
bx['side'] = np.sqrt(bx.w * bx.W * bx.h * bx.H)          # yazılan görüntüdeki kenar (px)

summary = df.groupby(['split', 'source']).agg(görüntü=('uid', 'size'),
                                              negatif=('cat', lambda c: int((c == 'negatif').sum())),
                                              fire_kutu=('n_fire', 'sum'),
                                              smoke_kutu=('n_smoke', 'sum'))
summary = summary.reindex(['train', 'val', 'test'], level=0)
display(summary)

BINS, LABELS = [0, 8, 16, 32, 96, np.inf], ['<8', '8-16', '16-32', '32-96', '>96']
def size_table(d, imgsz):
    side = d.side * imgsz / d[['W', 'H']].max(axis=1)
    b = pd.cut(side, BINS, labels=LABELS, right=False).rename('kenar_px')
    t = pd.crosstab([d.source, d.name], b, dropna=False)
    t = t[t.sum(axis=1) > 0]
    return t.div(t.sum(axis=1), axis=0).mul(100).round(1).reindex(columns=LABELS, fill_value=0)

size_tables = {sz: size_table(bx[bx.split == 'train'], sz) for sz in [640, 1024, 1280]}
for sz, tb in size_tables.items():
    print(f'\ntrain, imgsz={sz} sonrası kutu kenarı (%):')
    display(tb)

In [ ]:
COLORS = {0: 'red', 1: 'cyan'}
def grid(sel, title, n=18, cols=6):
    sel = sel.sample(min(n, len(sel)), random_state=SEED)
    rows_ = (len(sel) + cols - 1) // cols
    fig, axes = plt.subplots(rows_, cols, figsize=(cols * 3.2, rows_ * 2.4), squeeze=False)
    for ax in axes.ravel():
        ax.axis('off')
    for ax, (_, r) in zip(axes.ravel(), sel.iterrows()):
        with Image.open(OUT / r.file) as im:
            W, H = im.size
            ax.imshow(im.convert('RGB'))
        for c, x, y, w, h in r.boxes:
            ax.add_patch(plt.Rectangle(((x - w / 2) * W, (y - h / 2) * H), w * W, h * H,
                                       fill=False, color=COLORS[c], lw=1.2))
        ax.set_title(f'{r.split} · {r.uid[:22]}', fontsize=7)
    fig.suptitle(title)
    plt.tight_layout(); plt.show()

grid(df[(df.source == 'fasdd') & (df.cat != 'negatif')], 'FASDD — pozitifler (kırmızı=fire, camgöbeği=smoke)')
grid(df[(df.source == 'boreal') & (df.cat != 'negatif')], 'Boreal — pozitifler')
grid(df[df.cat == 'negatif'], 'Negatifler (iki kaynak)', n=12)

## 8. Drive'a kaydet

Dataset sıkıştırılmadan zip'lenir (JPEG zaten sıkışık, böylece hızlı). Sonraki oturumlarda eğitim notebook'u bu zip'i `/content`'e açacak.

In [ ]:
lines = ['# Dataset v1 — firesmoke', '',
         f'- Görüntü: {len(df)} · grup: {n_groups} · MAX_SIDE: {MAX_SIDE} · THR_GROUP: {THR_GROUP} · '
         f'THR_FASDD: {THR_FASDD} · seed: {SEED}',
         f'- Atılan sıfır alanlı kutu: {dropped_total} · çıkarılan etiketsiz Boreal görüntüsü: {len(sus)}',
         f'- Sızıntı (test↔train/val), FASDD: ≤{THR_FASDD} bit: '
         f'{int(((leak.src_i == "fasdd") & (leak.dist <= THR_FASDD)).sum())} çift · ≤{THR_CHECK} bit: '
         f'{leak[leak.src_i == "fasdd"].i.nunique()} test görüntüsünün komşusu var (phash sınırı; 04 notebookunda DINOv2 ile temizlenecek)',
         f'- Sızıntı (test↔train/val), Boreal ≤{THR_FASDD} bit: '
         f'{int(((leak.src_i == "boreal") & (leak.dist <= THR_FASDD)).sum())} çift (farklı yangınlar)',
         f'- Boreal test olayı: {TEST_EVENT} · D-Fire: kullanılmadı',
         '', '## Split × kaynak', summary.to_markdown(),
         '', '## Split × kaynak × kategori', t.to_markdown()]
for sz, tb in size_tables.items():
    lines += ['', f'## train, imgsz={sz} sonrası kutu kenarı (%)', tb.to_markdown()]
report = '\n'.join(lines)
(OUT / 'README.md').write_text(report)
(REP / 'dataset_v1.md').write_text(report)
shutil.copy(OUT / 'manifest.csv', REP / 'dataset_v1_manifest.csv')

dst = ROOT / 'datasets' / 'firesmoke_v1.zip'
dst.parent.mkdir(parents=True, exist_ok=True)
tmp = Path('/content/firesmoke_v1.zip')
if tmp.exists():
    tmp.unlink()
t0 = time.time()
subprocess.run(['zip', '-r', '-0', '-q', str(tmp), OUT.name], cwd=OUT.parent, check=True)
shutil.copy(tmp, dst)
tmp.unlink()
print(f'{dst}  {dst.stat().st_size / 1e9:.2f} GB  ({time.time() - t0:.0f} sn)')
print(report)

**Kontrol noktası:** FASDD'de 8 bit sızıntısı 0, Heinola testte, rapor ve zip Drive'da. Sıradaki adım `04`: GPU oturumunda dataset açılır, DINOv2 ile kalan yakın kopyalar temizlenir, ardından baseline eğitim.